# <center> ISEL - CSM </center>
## <center>Semestre 2023/24 </center>
###  <center> Trabalho 4 - Codificação de Sinais Multimédia</center>

Trabalho realizado por:
* João Ramos, numero <b> 50730 </b>
* Miguel Alcobia, numero <b> 50746 </b>
* Fábio Pestana, numero <b> 50756 </b>

<b> Turma 42D </b> <br></br>
<b> Docente: Gonçalo Valadão </b>

### 1.
Este trabalho explora os princípios básicos da codificação de vídeo. Neste trabalho pretende-se implementar três formas
de codificação de vídeo. Cada um destes codificadores deve ser testado com as sequências de imagens disponibilizadas
(``bola_seq.zip`` e ``carro_seq.zip``). Para cada codificador deve medir (frame a frame):<br><br>
    1. a taxa de compressão;<br>
    2. a relação sinal-ruído (relação entre a frame original no emissor e a frame final no recetor);<br>
    3. a entropia da frame a transmitir;<br>
    4. a energia média por pixel da frame a transmitir;<br>
    5. o tempo de compressão e descompressão.<br><br>
No final deve construir gráficos ou tabelas que ilustrem estas medidas em função da frame

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from time import time
import cv2 as cv
import math
import os

# Funções dos trabalhos anteriores

def calc_snr(img_org, img_app):
    """SNR - Signal-to-Noise Ratio

    Args:
        img_org: Imagem original
        img_app: Imagem com ruído

    Returns:
        snr_value: Valor do SNR
    """
    # Conversão dos valores da imagem para float
    img_org = img_org * 1.0
    img_app = img_app * 1.0

    # Calcular o sinal da imagem original
    signal = np.mean(img_org ** 2)
    
    # Calcular o ruído como a diferença entre as duas imagens (Mean-Square Error)
    noise = np.mean((img_org - img_app) ** 2)

    # Calcular a SNR (dB)
    snr_value = 10 * np.log10(signal / noise)
    return round(snr_value, 2)

# Taxa de Compressao (TP1 com alguns ajustes):
#   tamanho Imagem Original / tamanho Imagem alterada
def taxaCompressao(img_org_size, img_app_size):
    """Taxa de Compressão

    Calcula a taxa de compressão entre o tamanho da imagem original e a imagem alterada

    Args:
        img_org_size: Tamanho da imagem original
        img_app_size: Tamanho da imagem alterada
    Returns:
        factor_qualidade: Taxa de compressão
    """
    factor_qualidade = img_org_size/img_app_size
    return np.round(factor_qualidade)

# Calcular probabilidade

def count_freq_simb(frame):
    """Contar frequência por simbolos

    Args:
        frame: frame

    Returns:
        simbs, freqs: lista de simbolos no frame, lista da respetiva frequência
    """
    if not isinstance(frame, list):
        frame = frame.tolist()
        
    # Ver a frequência dos simbolos
    frequencia = {}
    for simb in frame:
        if simb in frequencia:
            frequencia[simb] += 1
        else:
            frequencia[simb] = 1
    
    simbs = []
    freqs = []

    for simb, freq in frequencia.items():
        simbs.append(simb)
        freqs.append(freq)

    return simbs, freqs

def calc_prob(simbs, freqs):
    """Calcula Probabilidade

    Args:
        simbs: lista de simbolos
        freqs: lista de frequências

    Returns:
        probs: array com as probabilidades
    """
    probs = []
    for i in range(len(simbs)):
        prob = freqs[i]/np.sum(np.fromiter((f for f in freqs), dtype=float))
        probs.append(prob)
    return probs

def calc_entropy(probs):
    """Calcula Entropia

    Args:
        probs: lista com probabilidade

    Returns:
        entropy: valor de entropia
    """
    entropy = -1*np.sum(np.fromiter((p * math.log2(p) for p in probs), dtype=float))
    return entropy

def calc_energia_media_pixel(frame):
    """Calcula Energia Média por Pixel

    Args:
        frame: Frame a analisar

    Returns:
        _type_: _description_
    """
    energia = np.square(frame).mean()
    return np.round(energia, 2)


In [ ]:
# Sequências de imagens
frames_bola = ["bola_" + str(i) + ".tiff" for i in range(1, 12)]
frames_carro = ["carro_" + str(i) + ".tiff" for i in range(1, 12)]

###################################
# Plot das imagens da bola
###################################

fig, axs = plt.subplots(2, 6, figsize=(15, 5))
fig.suptitle("Frames da Bola", fontsize=16)

for i, image_file in enumerate(frames_bola):
    row = i // 6
    col = i % 6
    img = cv.imread(image_file)  # Carregar a imagem com OpenCV
    img = cv.cvtColor(img, cv.COLOR_BGR2RGB)  # Converter de BGR para RGB
    axs[row, col].imshow(img)
    axs[row, col].axis('off')  # Esconder os eixos

# Desativar os eixos extras
for j in range(len(frames_bola), 12):
    row = j // 6
    col = j % 6
    axs[row, col].axis('off')

plt.show()

###################################
# Plot das imagens do carro
##################################

fig, axs = plt.subplots(2, 6, figsize=(15, 5))
fig.suptitle("Frames do Carro", fontsize=16)

for i, image_file in enumerate(frames_carro):
    row = i // 6
    col = i % 6
    img = cv.imread(image_file)  # Carregar a imagem com OpenCV
    img = cv.cvtColor(img, cv.COLOR_BGR2RGB)  # Converter de BGR para RGB
    axs[row, col].imshow(img)
    axs[row, col].axis('off')  # Esconder os eixos

# Desativar os eixos extras
for j in range(len(frames_carro), 12):
    row = j // 6
    col = j % 6
    axs[row, col].axis('off')

plt.show()

Para os três codificadores use o algoritmo JPEG do OpenCV com uma qualidade de 50%.
As três formas de codificação são:
1. Considerar que cada frame é uma intra-frame (I).
2. Considerar que todas as frames à exceção da primeira (a I-frame) são inter-frames (P). Neste codificador deve
criar as P-frames, que são a diferença entre a frame a codificar e a I-frame, sem compensação de movimento.
Visualize a P-frame (ou seja a imagem a transmitir).
3. Considerar que todas as frames à exceção da primeira são inter-frames (P). Neste codificador deve implementar
a predição da frame a codificar com base na I-frame fazendo a compensação de movimento. A frame a transmitir
é a diferença entre a frame a codificar e a sua predição. Sugere-se a construção de três funções:<br>
3.1. uma função que realiza a medição do erro absoluto médio entre dois blocos (tamanho 16 × 16);<br>
3.2. uma função que realiza uma pesquisa (pode escolher a full-search ou outra) do macro-bloco da frame a
codificar numa janela de pesquisa (-15 a +15) da I-frame;<br>
3.3. uma função que percorra os blocos da frame a codificar e construa a frame predita;<br>
Visualizar a frame predita, e a frame diferença, bem como os vetores de movimento (use a função pylab.quiver
para o efeito).

1. Considerar que cada frame é uma intra-frame (I).

In [ ]:
# Função para codificar a imagem como JPG com qualidade de 50%
def encode_jpg(pasta, image, nome, i, pb=1):
    """Encode JPG

    Args:
        pasta: pasta para guardar a imagem
        image: imagem a codificar
        nome: nome a dar ao ficheiro
        i: indice da frame
        pb: print boolean para imprimir valores. Defaults to 1.

    Returns:
        Imagem em JPG e respetivas medidas
    """

    # Codificação
    t0 = time()
    encode_param = [int(cv.IMWRITE_JPEG_QUALITY), 50]
    _, encoded_image = cv.imencode('.jpg', image, encode_param)
    t1 = time()
    tComp = t1 - t0

    # Descodificação
    t2 = time()
    decoded_image = cv.imdecode(encoded_image, cv.IMREAD_COLOR)
    decoded_imageBW = cv.imdecode(encoded_image, cv.IMREAD_GRAYSCALE) # Para retirar o color channel e facilitar as medidas
    t3 = time()
    tDecomp = t3 - t2

    # Guardar Imagem
    img_name = f"{pasta}/{nome}_{i}.jpg"
    cv.imwrite(img_name, decoded_image)

    # Medidas
    original_size = os.path.getsize(f"{nome}_{i+1}.tiff")
    compressed_size = os.path.getsize(img_name)

    snr_frame = calc_snr(image, decoded_image)
    txcomp = taxaCompressao(original_size, compressed_size)

    simbs, freqs = count_freq_simb(decoded_imageBW.ravel())
    probs = calc_prob(simbs, freqs)
    entropia = calc_entropy(probs)
    emp = calc_energia_media_pixel(decoded_imageBW)
    
    if (pb == 1):
        print("SNR:", snr_frame)
        print("Taxa de Compressão:", txcomp)
        print("Entropia:", entropia)
        print("Energia Média por pixel:", emp)
        print("\n")
        
    return decoded_image, snr_frame, txcomp, entropia, emp, tComp, tDecomp

In [ ]:
###################################
# Plot das imagens da bola
###################################

fig, axs = plt.subplots(2, 6, figsize=(15, 5))
fig.suptitle("Frames da Bola", fontsize=16)
plt.subplots_adjust(wspace=0.1, hspace=0.1)  # Ajustar o espaço entre as imagens

Frames_IDX = []
SNRs_Bola = []
TaxaComp_Bola = []
Energia_Med_Pixel_Bola = []
Entropia_Bola = []
TimesComp_Bola = []
TimesDecomp_Bola = []

for i, image_file in enumerate(frames_bola):
    print(image_file)
    row = i // 6
    col = i % 6
    img = cv.imread(image_file)  # Carregar a imagem com OpenCV
    img, snr_frame, txcomp, entropia, emp, tComp, tDecomp = encode_jpg("img_ex1", img, "bola", i)  # Codificar como JPG com qualidade de 50%
    
    Frames_IDX.append(i)
    SNRs_Bola.append(snr_frame)
    TaxaComp_Bola.append(txcomp)
    Energia_Med_Pixel_Bola.append(emp)
    Entropia_Bola.append(entropia)
    TimesComp_Bola.append(tComp)
    TimesDecomp_Bola.append(tDecomp)

    img = cv.cvtColor(img, cv.COLOR_BGR2RGB)  # Converter de BGR para RGB
    axs[row, col].imshow(img)
    axs[row, col].axis('off')  # Esconder os eixos

# Desativar os eixos extras
for j in range(len(frames_bola), 12):
    row = j // 6
    col = j % 6
    axs[row, col].axis('off')

plt.show()

# Tabela

print("|    Frames    |    SNR       | Taxa de Comp | Energia Med Pixel |  Entropia    |  Tempo (Comp) (s)  |  Tempo (Descomp) (s)  |")
print("|--------------|--------------|--------------|-------------------|--------------|--------------------|-----------------------|")

# Preenchendo a tabela com os valores dos arrays
for frames_idx, snr_tab, taxa, emp, entropia, tempoC, tempoD in zip(Frames_IDX, SNRs_Bola, TaxaComp_Bola, Energia_Med_Pixel_Bola, Entropia_Bola, TimesComp_Bola, TimesDecomp_Bola):
    frame_str = f"{frames_idx+1}".center(12)
    snr_str = f"{snr_tab:.2f}".center(12)
    taxa_str = f"{taxa:.1f}".center(12)
    emp_str = f"{emp:.1f}".center(17)
    entropia_str = f"{entropia:.1f}".center(12)
    tempoC_str = f"{tempoC:.7f}".center(18)
    tempoD_str = f"{tempoD:.7f}".center(21)
    print(f"| {frame_str} | {snr_str} | {taxa_str} | {emp_str} | {entropia_str} | {tempoC_str} | {tempoD_str} |")

print("\n\n")
    

###################################
# Plot das imagens do carro
###################################

fig, axs = plt.subplots(2, 6, figsize=(15, 5))
fig.suptitle("Frames do Carro", fontsize=16)
plt.subplots_adjust(wspace=0.1, hspace=0.1)  # Ajustar o espaço entre as imagens

Frames_IDX = []
SNRs_Carro = []
TaxaComp_Carro = []
Energia_Med_Pixel_Carro = []
Entropia_Carro = []
TimesComp_Carro = []
TimesDecomp_Carro = []

for i, image_file in enumerate(frames_carro):
    print(image_file)
    row = i // 6
    col = i % 6
    img = cv.imread(image_file)  # Carregar a imagem com OpenCV
    img, snr_frame, txcomp, entropia, emp, tComp, tDecomp = encode_jpg("img_ex1", img, "carro", i)  # Codificar como JPEG com qualidade de 50%
    
    Frames_IDX.append(i)
    SNRs_Carro.append(snr_frame)
    TaxaComp_Carro.append(txcomp)
    Energia_Med_Pixel_Carro.append(emp)
    Entropia_Carro.append(entropia)
    TimesComp_Carro.append(tComp)
    TimesDecomp_Carro.append(tDecomp)
    
    img = cv.cvtColor(img, cv.COLOR_BGR2RGB)  # Converter de BGR para RGB
    axs[row, col].imshow(img)
    axs[row, col].axis('off')  # Esconder os eixos

# Desativar os eixos extras
for j in range(len(frames_carro), 12):
    row = j // 6
    col = j % 6
    axs[row, col].axis('off')

plt.show()

# Tabela

print("|    Frames    |    SNR       | Taxa de Comp | Energia Med Pixel |  Entropia    |  Tempo (Comp) (s)  |  Tempo (Descomp) (s)  |")
print("|--------------|--------------|--------------|-------------------|--------------|--------------------|-----------------------|")

# Preenchendo a tabela com os valores dos arrays
for frames_idx, snr, taxa, emp, entropia, tempoC, tempoD in zip(Frames_IDX, SNRs_Carro, TaxaComp_Carro, Energia_Med_Pixel_Carro, Entropia_Carro, TimesComp_Carro, TimesDecomp_Carro):
    frame_str = f"{frames_idx+1}".center(12)
    snr_str = f"{snr:.2f}".center(12)
    taxa_str = f"{taxa:.1f}".center(12)
    emp_str = f"{emp:.1f}".center(17)
    entropia_str = f"{entropia:.1f}".center(12)
    tempoC_str = f"{tempoC:.7f}".center(18)
    tempoD_str = f"{tempoD:.7f}".center(21)
    print(f"| {frame_str} | {snr_str} | {taxa_str} | {emp_str} | {entropia_str} | {tempoC_str} | {tempoD_str} |")

print("\n\n")

2. Considerar que todas as frames à exceção da primeira (a I-frame) são inter-frames (P). Neste codificador deve
criar as P-frames, que são a diferença entre a frame a codificar e a I-frame, sem compensação de movimento.
Visualize a P-frame (ou seja a imagem a transmitir).

In [ ]:
# Função para calcular a P-frame (diferença com a I-frame) e respetiva descodificação
def calculate_p_frame(frame_atual, i_frame):
    """Calcular as P-Frames

    Args:
        frame_atual (_type_): _description_
        i_frame (_type_): _description_

    Returns:
        diferença entre as frames
    """
    
    #diff = cv.absdiff(current_frame, i_frame)
    
    # Devido ao facto do OpenCv não aceitar valores negativos, soma-se 255 e dividimos por 2
    diff = (frame_atual - i_frame + 255) // 2
    return diff

def decode_p_frame(p_frame, i_frame):
    """Descodificar P-Frames

    Args:
        p_frame: P-Frames_
        i_frame: I-Frames

    Returns:
        Frame original, obtida a partir da diferença
    """
    decode_frame = p_frame * 2 - 255 + i_frame
    
    #decode_frame = -1 * p_frame + i_frame
    return decode_frame
    


###################################
# PLOT DOS FRAMES DA BOLA
###################################

# Codificar a I-frame
t0 = time()
i_frame_bola = cv.imread(frames_bola[0])
i_frame_bola_encoded = encode_jpg("img_ex2_1", i_frame_bola, "bola", 0, 0)[0]

fig, axs = plt.subplots(2, 6, figsize=(15, 5))
fig.suptitle("Frames da Bola (P-frames) Codificadas", fontsize=16)
plt.subplots_adjust(wspace=0.1, hspace=0.1)  # Ajustar o espaço entre as imagens

# Exibir a I-frame
axs[0, 0].imshow(cv.cvtColor(i_frame_bola_encoded, cv.COLOR_BGR2RGB))
axs[0, 0].axis('off')

p_frames = []

Frames_IDX = []
SNRs_Bola = []
TaxaComp_Bola = []
Energia_Med_Pixel_Bola = []
Entropia_Bola = []
TimesComp_Bola = []
TimesDecomp_Bola = []

# Calcular e exibir as P-frames
for i, image_file in enumerate(frames_bola[1:]):
    row = (i + 1) // 6
    col = (i + 1) % 6
    current_frame = cv.imread(image_file)  # Carregar a imagem com OpenCV
    current_frame_encoded = encode_jpg("img_ex2_1", current_frame,"bola", i+1, 0)[0]  # Codificar como JPG com qualidade de 50%
    p_frame = calculate_p_frame(current_frame_encoded, i_frame_bola_encoded)  # Calcular a P-frame (codificação)
    t1 = time()
    tComp = t1 - t0

    t2 = time()
    p_frame_decode = decode_p_frame(p_frame, i_frame_bola_encoded)
    t3 = time()
    tDecomp = t3 - t2
    p_frames.append(p_frame_decode)

    axs[row, col].imshow(cv.cvtColor(p_frame, cv.COLOR_BGR2RGB))  # Converter de BGR para RGB
    axs[row, col].axis('off')  # Esconder os eixos

    # Guardar a imagem da P-Frame
    img_name = f"img_ex2_2/bola_{i}.jpg"
    cv.imwrite(img_name, p_frame)

    # Medidas
    original_size = os.path.getsize(f"bola_{i+1}.tiff")
    compressed_size = os.path.getsize(img_name)

    snr_frame = calc_snr(current_frame, p_frame_decode)
    print("SNR:", snr_frame)

    txComp = taxaCompressao(original_size, compressed_size)
    print("Taxa de Compressão:", txComp)

    simbs, freqs = count_freq_simb(p_frame.ravel())
    probs = calc_prob(simbs, freqs)
    entropia = calc_entropy(probs)
    print("Entropia:", entropia)

    emp = calc_energia_media_pixel(p_frame)
    print("Energia Média por pixel:", emp)

    Frames_IDX.append(i)
    SNRs_Bola.append(snr_frame)
    TaxaComp_Bola.append(txcomp)
    Energia_Med_Pixel_Bola.append(emp)
    Entropia_Bola.append(entropia)
    TimesComp_Bola.append(tComp)
    TimesDecomp_Bola.append(tDecomp)

    print("\n")

#print("FramesIDX",len(Frames_IDX))

# Desativar os eixos
for j in range(len(frames_bola), 12):
    row = j // 6
    col = j % 6
    axs[row, col].axis('off')

plt.show()

print("|    Frames    |    SNR       | Taxa de Comp | Energia Med Pixel |  Entropia    |  Tempo (Comp) (s)  |  Tempo (Descomp) (s)  |")
print("|--------------|--------------|--------------|-------------------|--------------|--------------------|-----------------------|")

for frames_idx, snr_tab, taxa, emp, entropia, tempoC, tempoD in zip(Frames_IDX, SNRs_Bola, TaxaComp_Bola, Energia_Med_Pixel_Bola, Entropia_Bola, TimesComp_Bola, TimesDecomp_Bola):
    frame_str = f"{frames_idx+2}".center(12)
    snr_str = f"{snr_tab:.2f}".center(12)
    taxa_str = f"{taxa:.1f}".center(12)
    emp_str = f"{emp:.1f}".center(17)
    entropia_str = f"{entropia:.1f}".center(12)
    tempoC_str = f"{tempoC:.7f}".center(18)
    tempoD_str = f"{tempoD:.7f}".center(21)
    print(f"| {frame_str} | {snr_str} | {taxa_str} | {emp_str} | {entropia_str} | {tempoC_str} | {tempoD_str} |")

print("\n\n")

# Plot descodificado

fig, axs = plt.subplots(2, 6, figsize=(15, 5))
fig.suptitle("Frames da Bola (P-frames) Descodificadas", fontsize=16)
plt.subplots_adjust(wspace=0.1, hspace=0.1)  # Ajustar o espaço entre as imagens
# Exibir a I-frame
axs[0, 0].imshow(cv.cvtColor(i_frame_bola_encoded, cv.COLOR_BGR2RGB))
axs[0, 0].axis('off')

# Calcular e exibir as P-frames
for i in range(len(p_frames)):
    row = (i + 1) // 6
    col = (i + 1) % 6
    p_frame_decode = p_frames[i]
    axs[row, col].imshow(cv.cvtColor(p_frame_decode.astype(np.uint8), cv.COLOR_BGR2RGB))  # Converter de BGR para RGB
    axs[row, col].axis('off')  # Esconder os eixos

# Desativar os eixos extras
for j in range(len(frames_carro), 12):
    row = j // 6
    col = j % 6
    axs[row, col].axis('off')

plt.show()

In [ ]:
###################################
# PLOT DOS FRAMES DO CARRO
###################################

# Codificar a I-frame
t0 = time()
i_frame_carro = cv.imread(frames_carro[0])
i_frame_carro_encoded = encode_jpg("img_ex2_1",i_frame_carro, "carro", 0, 0)[0]

fig, axs = plt.subplots(2, 6, figsize=(15, 5))
fig.suptitle("Frames do Carro (P-frames) Codificadas", fontsize=16)
plt.subplots_adjust(wspace=0.1, hspace=0.1)  # Ajustar o espaço entre as imagens

# Exibir a I-frame
axs[0, 0].imshow(cv.cvtColor(i_frame_carro_encoded, cv.COLOR_BGR2RGB))
axs[0, 0].axis('off')

Frames_IDX = []
SNRs_Carro = []
TaxaComp_Carro = []
Energia_Med_Pixel_Carro = []
Entropia_Carro = []
TimesComp_Carrp = []
TimesDecomp_Carro = []

p_frames = []
# Calcular e exibir as P-frames
for i, image_file in enumerate(frames_carro[1:]):
    row = (i + 1) // 6
    col = (i + 1) % 6
    current_frame = cv.imread(image_file)  # Carregar a imagem com OpenCV
    current_frame_encoded = encode_jpg("img_ex2_1", current_frame, "carro", i, 0)[0]  # Codificar como JPEG com qualidade de 50%
    p_frame = calculate_p_frame(current_frame_encoded,i_frame_carro_encoded)  # Calcular a P-frame (Codificação)
    t1 = time()
    tComp = t1- t0

    t2 = time()
    p_frame_decode = decode_p_frame(p_frame, i_frame_carro_encoded) # Descodificação
    p_frames.append(p_frame_decode)
    t3 = time()
    tDecomp = t3 - t2

    axs[row, col].imshow(cv.cvtColor(p_frame, cv.COLOR_BGR2RGB))  # Converter de BGR para RGB
    axs[row, col].axis('off')  # Esconder os eixos

    
    img_name = f"img_ex2_2/carro_{i}.jpg"
    cv.imwrite(img_name, p_frame)

    original_size = os.path.getsize(f"bola_{i+1}.tiff")
    compressed_size = os.path.getsize(img_name)

    snr_frame = calc_snr(current_frame, p_frame_decode)
    print("SNR:", snr_frame)

    txComp = taxaCompressao(original_size, compressed_size)
    print("Taxa de Compressão:", txComp)

    simbs, freqs = count_freq_simb(p_frame.ravel())
    probs = calc_prob(simbs, freqs)
    entropia = calc_entropy(probs)
    print("Entropia:", entropia)

    emp = calc_energia_media_pixel(p_frame)
    print("Energia Média por pixel:", emp)

    Frames_IDX.append(i)
    SNRs_Carro.append(snr_frame)
    TaxaComp_Carro.append(txcomp)
    Energia_Med_Pixel_Carro.append(emp)
    Entropia_Carro.append(entropia)
    TimesComp_Carro.append(tComp)
    TimesDecomp_Carro.append(tDecomp)

    print("\n")

#print("FramesIDX",len(Frames_IDX))

# Desativar os eixos
for j in range(len(frames_carro), 12):
    row = j // 6
    col = j % 6
    axs[row, col].axis('off')

plt.show()

print("|    Frames    |    SNR       | Taxa de Comp | Energia Med Pixel |  Entropia    |  Tempo (Comp) (s)  |  Tempo (Descomp) (s)  |")
print("|--------------|--------------|--------------|-------------------|--------------|--------------------|-----------------------|")

for frames_idx, snr_tab, taxa, emp, entropia, tempoC, tempoD in zip(Frames_IDX, SNRs_Carro, TaxaComp_Carro, Energia_Med_Pixel_Carro, Entropia_Carro, TimesComp_Carro, TimesDecomp_Carro):
    frame_str = f"{frames_idx+2}".center(12)
    snr_str = f"{snr_tab:.2f}".center(12)
    taxa_str = f"{taxa:.1f}".center(12)
    emp_str = f"{emp:.1f}".center(17)
    entropia_str = f"{entropia:.1f}".center(12)
    tempoC_str = f"{tempoC:.7f}".center(18)
    tempoD_str = f"{tempoD:.7f}".center(21)
    print(f"| {frame_str} | {snr_str} | {taxa_str} | {emp_str} | {entropia_str} | {tempoC_str} | {tempoD_str} |")

print("\n\n")

# Plot descodificado

fig, axs = plt.subplots(2, 6, figsize=(15, 5))
fig.suptitle("Frames do Carro (P-frames) Descodificadas", fontsize=16)
plt.subplots_adjust(wspace=0.1, hspace=0.1)  # Ajustar o espaço entre as imagens
# Exibir a I-frame
axs[0, 0].imshow(cv.cvtColor(i_frame_carro_encoded, cv.COLOR_BGR2RGB))
axs[0, 0].axis('off')

# Calcular e exibir as P-frames
for i in range(len(p_frames)):
    row = (i + 1) // 6
    col = (i + 1) % 6
    p_frame_decode = p_frames[i]
    axs[row, col].imshow(cv.cvtColor(p_frame_decode.astype(np.uint8), cv.COLOR_BGR2RGB))  # Converter de BGR para RGB
    axs[row, col].axis('off')  # Esconder os eixos

# Desativar os eixos extras
for j in range(len(frames_carro), 12):
    row = j // 6
    col = j % 6
    axs[row, col].axis('off')

plt.show()

3. Considerar que todas as frames à exceção da primeira são inter-frames (P). Neste codificador deve implementar
a predição da frame a codificar com base na I-frame fazendo a compensação de movimento. A frame a transmitir
é a diferença entre a frame a codificar e a sua predição. Sugere-se a construção de três funções:<br><br>
3.1. uma função que realiza a medição do erro absoluto médio entre dois blocos (tamanho 16 × 16);<br>

In [ ]:
def calcular_mae(bloco1, bloco2):
    """
    Calcula o Erro Absoluto Médio (Mean Absolute Error - MAE) entre dois blocos de 16x16 pixels.
    
    Args:
    bloco1
    bloco2
    
    Returns:
    float: O erro absoluto médio entre os dois blocos.
    """
    
    # Calcular o erro absoluto médio
    mae = np.mean(np.abs(bloco1 - bloco2))
    return mae

3.2. uma função que realiza uma pesquisa (pode escolher a full-search ou outra) do macro-bloco da frame a
codificar numa janela de pesquisa (-15 a +15) da I-frame;<br>

In [ ]:
def pesquisar_macrobloco(frame_atual, frame_referencia, x, y):
    """
    Pesquisa o macro-bloco (16x16)

    Args:
        frame_atual: Frame atual a ser codificada.
        frame_referencia: I-frame de referência.
        x: Coordenada x do canto superior esquerdo do bloco na frame atual.
        y: Coordenada y do canto superior esquerdo do bloco na frame atual.

    Returns:
        A posição (dx, dy) do melhor bloco correspondente na I-frame.
    """
    bloco_atual = frame_atual[y:y+16, x:x+16]
    melhor_mae = float('inf') # Começar com um MAE mínimo com infinito
    melhor_dx, melhor_dy = 0, 0
    
    # Definir limites da janela de pesquisa
    for dy in range(-15, 16):
        for dx in range(-15, 16):
            x_ref = x + dx # Calcular a posição x do bloco na I-Frame
            y_ref = y + dy # Calcular a posição y do bloco na I-Frame
            
            # Verificar se os índices estão dentro dos limites da frame de referência
            if (x_ref >= 0 and x_ref + 15 < frame_referencia.shape[1] and
                y_ref >= 0 and y_ref + 15 < frame_referencia.shape[0]):
                # Extrair o bloco 16x16 da I-Frame
                bloco_referencia = frame_referencia[y_ref:y_ref+16, x_ref:x_ref+16]
                mae = calcular_mae(bloco_atual, bloco_referencia)
                
                # Atualizar a melhor correspondência
                if mae < melhor_mae:
                    melhor_mae = mae
                    melhor_dx, melhor_dy = dx, dy # Atualizar os melhores deslocamentos dx e dy
    
    return melhor_dx, melhor_dy

3.3. uma função que percorra os blocos da frame a codificar e construa a frame predita;<br>
Visualizar a frame predita, e a frame diferença, bem como os vetores de movimento (use a função pylab.quiver
para o efeito).

In [ ]:

def construir_frame_predita(frame_atual, frame_referencia):
    """
    Constrói a frame predita para a frame atual baseada na I-Frame.

    Args:
        frame_atual: Frame atual a ser codificada.
        frame_referencia: I-Frame.

    Returns:
        A frame predita e a diferença entre a frame atual e a I-Frame.
    """
    altura, largura = frame_atual.shape
    frame_predita = np.zeros_like(frame_atual)
    diferenca = np.zeros_like(frame_atual)
    
    # Percorrer todos os blocos 16x16
    for y in range(0, altura, 16):
        for x in range(0, largura, 16):
            dx, dy = pesquisar_macrobloco(frame_atual, frame_referencia, x, y)
            x_ref = x + dx
            y_ref = y + dy
            
            # Extrair o bloco correspondente na I-frame
            bloco_referencia = frame_referencia[y_ref:y_ref+16, x_ref:x_ref+16]
            
            # Preencher a frame predita
            frame_predita[y:y+16, x:x+16] = bloco_referencia
            
            # Calcular a diferença
            diferenca[y:y+16, x:x+16] = frame_atual[y:y+16, x:x+16] - bloco_referencia
    
    return frame_predita, diferenca

> Testes

In [ ]:
frames = []

# Carregar os frames a partir dos arquivos .tiff
for i in range(1, 12):
    frame = cv.imread(f'bola_{i}.tiff', cv.IMREAD_GRAYSCALE)
    #print("OG", frame.shape)
    """cv.imshow("Teste", frame)
    cv.waitKey(0) 
  
    # closing all open windows 
    cv.destroyAllWindows() """
    frames.append(frame)

# Vamos supor que a primeira frame seja a I-frame de referência
frame_referencia = frames[0]

# Criar uma lista para armazenar as frames preditas e as diferenças
frames_preditas = []
diferencas = []
vetores_movimento = []

Frames_IDX = []
SNRs_Bola = []
TaxaComp_Bola = []
Energia_Med_Pixel_Bola = []
Entropia_Bola = []
TimesComp_Bola = []
TimesDecomp_Bola = []

# Loop sobre os frames para aplicar a compensação de movimento
for i in range(1, len(frames)):
    t0 = time()
    frame_atual = frames[i]
    #print("FRAME", frame_atual.shape)
    
    # Construir a frame predita e calcular a diferença
    frame_predita, diferenca = construir_frame_predita(frame_atual, frame_referencia)
    
    # Adicionar as frames preditas e as diferenças às listas correspondentes
    frames_preditas.append(frame_predita)
    diferencas.append(diferenca)

    diff_name = f"img_ex3_3/bola_diff_{i}.jpg"
    cv.imwrite(diff_name, diferenca)
    
    # Calcular os vetores de movimento
    altura, largura = frame_atual.shape
    vetores = np.zeros((altura // 16, largura // 16, 2))  # Vetores de movimento para cada bloco 16x16
    
    for y in range(0, altura, 16):
        for x in range(0, largura, 16):
            dx, dy = pesquisar_macrobloco(frame_atual, frame_referencia, x, y)
            vetores[y // 16, x // 16] = [dx, dy]
    
    img_name = f"img_ex3_3/bola_{i}.jpg"
    cv.imwrite(img_name, frame_predita)

    t1 = time()
    tComp = t1 - t0

    original_size = os.path.getsize(f"bola_{i+1}.tiff")
    compressed_size = os.path.getsize(img_name)

    print("Frame ", i+1)
    snr_frame = calc_snr(frame_atual, cv.imread(img_name, cv.IMREAD_GRAYSCALE))
    print("SNR:", snr_frame)
    txComp = taxaCompressao(original_size, compressed_size)
    print("Taxa de Compressão:", txComp)

    simbs, freqs = count_freq_simb(frame_predita.ravel())
    probs = calc_prob(simbs, freqs)
    entropia = calc_entropy(probs)
    print("Entropia:", entropia)

    emp = calc_energia_media_pixel(p_frame)
    print("Energia Média por pixel:", emp)
    print("\n")

    vetores_movimento.append(vetores)

    Frames_IDX.append(i)
    SNRs_Bola.append(snr_frame)
    TaxaComp_Bola.append(txcomp)
    Energia_Med_Pixel_Bola.append(emp)
    Entropia_Bola.append(entropia)
    TimesComp_Bola.append(tComp)
    TimesDecomp_Bola.append(tDecomp)

# Visualização das frames preditas, diferenças e vetores de movimento
for i in range(len(frames_preditas)):
    plt.figure(figsize=(12, 4))
    
    # Frame original
    plt.subplot(131)
    plt.imshow(frames[i + 1], cmap='gray')
    plt.title('Frame Original')
    
    # Frame predita
    plt.subplot(132)
    plt.imshow(frames_preditas[i], cmap='gray')
    plt.title('Frame Predita')
    
    # Frame de diferença
    plt.subplot(133)
    plt.imshow(diferencas[i], cmap='gray')
    plt.title('Frame de Diferença')
    
    # Vetores de movimento
    plt.figure(figsize=(15, 15))
    plt.imshow(frames[i + 1], cmap='gray')
    plt.title('Vetores de Movimento')
    for y in range(0, vetores_movimento[i].shape[0]):
        for x in range(0, vetores_movimento[i].shape[1]):
            dx, dy = vetores_movimento[i][y, x]
            plt.quiver(x * 16 + 8, y * 16 + 8, dx, dy, color='red', angles='xy', scale_units='xy', scale=1)
    plt.show()

print("|    Frames    |    SNR       | Taxa de Comp | Energia Med Pixel |  Entropia    |  Tempo (Comp) (s)  |")
print("|--------------|--------------|--------------|-------------------|--------------|--------------------|")

# Preenchendo a tabela com os valores dos arrays
for frames_idx, snr_tab, taxa, emp, entropia, tempoC in zip(Frames_IDX, SNRs_Bola, TaxaComp_Bola, Energia_Med_Pixel_Bola, Entropia_Bola, TimesComp_Bola):
    frame_str = f"{frames_idx}".center(12)
    snr_str = f"{snr_tab:.2f}".center(12)
    taxa_str = f"{taxa:.1f}".center(12)
    emp_str = f"{emp:.1f}".center(17)
    entropia_str = f"{entropia:.1f}".center(12)
    tempoC_str = f"{tempoC:.7f}".center(18)
    print(f"| {frame_str} | {snr_str} | {taxa_str} | {emp_str} | {entropia_str} | {tempoC_str} |")

print("\n\nFim da Tabela")

In [ ]:
frames = []

# Carregar os frames a partir dos arquivos .tiff
for i in range(1, 12):
    frame = cv.imread(f'carro_{i}.tiff', cv.IMREAD_GRAYSCALE)
    """
    cv.imshow("Teste", frame)
    cv.waitKey(0) 
  
    # closing all open windows 
    cv.destroyAllWindows() 
    """
    frames.append(frame)

# Vamos supor que a primeira frame seja a I-frame de referência
frame_referencia = frames[0]

# Criar uma lista para armazenar as frames preditas e as diferenças
frames_preditas = []
diferencas = []
vetores_movimento = []


Frames_IDX = []
SNRs_Carro = []
TaxaComp_Carro = []
Energia_Med_Pixel_Carro = []
Entropia_Carro = []
TimesComp_Carro = []
TimesDecomp_Carro = []

# Loop sobre os frames para aplicar a compensação de movimento
for i in range(1, len(frames)):
    t0 = time()
    frame_atual = frames[i]
    
    # Construir a frame predita e calcular a diferença
    frame_predita, diferenca = construir_frame_predita(frame_atual, frame_referencia)
    
    # Adicionar as frames preditas e as diferenças às listas correspondentes
    frames_preditas.append(frame_predita)
    diferencas.append(diferenca)

    diff_name = f"img_ex3_3/carro_diff_{i}.jpg"
    cv.imwrite(diff_name, diferenca)
    
    # Calcular os vetores de movimento
    altura, largura = frame_atual.shape
    vetores = np.zeros((altura // 16, largura // 16, 2))  # Vetores de movimento para cada bloco 16x16

    for y in range(0, altura, 16):
        for x in range(0, largura, 16):
            dx, dy = pesquisar_macrobloco(frame_atual, frame_referencia, x, y)
            vetores[y // 16, x // 16] = [dx, dy]
    

    img_name = f"img_ex3_3/carro_{i}.jpg"
    cv.imwrite(img_name, frame_predita)

    t1 = time()
    tComp = t1-t0

    original_size = os.path.getsize(f"carro_{i+1}.tiff")
    compressed_size = os.path.getsize(img_name)

    print("Frame ", i+1)
    snr_frame = calc_snr(frame_atual, cv.imread(img_name, cv.IMREAD_GRAYSCALE))
    print("SNR:", snr_frame)
    txComp = taxaCompressao(original_size, compressed_size)
    print("Taxa de Compressão:", txComp)

    simbs, freqs = count_freq_simb(frame_predita.ravel())
    probs = calc_prob(simbs, freqs)
    entropia = calc_entropy(probs)
    print("Entropia:", entropia)
    
    emp = calc_energia_media_pixel(p_frame)
    print("Energia Média por pixel:", emp)
    print("\n")

    Frames_IDX.append(i)
    SNRs_Carro.append(snr_frame)
    TaxaComp_Carro.append(txcomp)
    Energia_Med_Pixel_Carro.append(emp)
    Entropia_Carro.append(entropia)
    TimesComp_Carro.append(tComp)
    TimesDecomp_Carro.append(tDecomp)

    vetores_movimento.append(vetores)

print("FRAMES", len(Frames_IDX))

# Visualização das frames preditas, diferenças e vetores de movimento
for i in range(len(frames_preditas)):
    plt.figure(figsize=(12, 4))
    
    # Frame original
    plt.subplot(131)
    plt.imshow(frames[i + 1], cmap='gray')
    plt.title('Frame Original')
    
    # Frame predita
    plt.subplot(132)
    plt.imshow(frames_preditas[i], cmap='gray')
    plt.title('Frame Predita')
    
    # Frame de diferença
    plt.subplot(133)
    plt.imshow(diferencas[i], cmap='gray')
    plt.title('Frame de Diferença')
    
    # Vetores de movimento
    plt.figure(figsize=(15, 15))
    plt.imshow(frames[i + 1], cmap='gray')
    plt.title('Vetores de Movimento')
    for y in range(0, vetores_movimento[i].shape[0]):
        for x in range(0, vetores_movimento[i].shape[1]):
            dx, dy = vetores_movimento[i][y, x]
            plt.quiver(x * 16 + 8, y * 16 + 8, dx, dy, color='red', angles='xy', scale_units='xy', scale=1)
    plt.show()

print("|    Frames    |    SNR       | Taxa de Comp | Energia Med Pixel |  Entropia    |  Tempo (Comp) (s)  |")
print("|--------------|--------------|--------------|-------------------|--------------|--------------------|")

# Preenchendo a tabela com os valores dos arrays
for frames_idx, snr_tab, taxa, emp, entropia, tempoC in zip(Frames_IDX, SNRs_Carro, TaxaComp_Carro, Energia_Med_Pixel_Carro, Entropia_Carro, TimesComp_Carro):
    frame_str = f"{frames_idx+1}".center(12)
    snr_str = f"{snr_tab:.2f}".center(12)
    taxa_str = f"{taxa:.1f}".center(12)
    emp_str = f"{emp:.1f}".center(17)
    entropia_str = f"{entropia:.1f}".center(12)
    tempoC_str = f"{tempoC:.7f}".center(18)
    print(f"| {frame_str} | {snr_str} | {taxa_str} | {emp_str} | {entropia_str} | {tempoC_str} |")

print("\n\nFim da Tabela")

> DESCODIFICADOR

In [ ]:
# Função para reconstruir a frame original a partir da frame predita e da diferença
def reconstruir_frame_original(frame_predita, diferenca):
    return frame_predita + diferenca

# Carregar a I-Frame
frame_referencia = cv.imread(f'bola_1.tiff', cv.IMREAD_GRAYSCALE)

frames_reconstruidas = []

Frames_IDX = []
TimesDecomp_Bola = []
# Loop para reconstruir as frames originais
for i in range(1, 11):

    frame_original = cv.imread(f'bola_{i+1}.tiff', cv.IMREAD_GRAYSCALE)

    # Carregar a frame predita
    Frames_IDX.append(i)
    t0 = time()
    frame_predita = cv.imread(f'img_ex3_3/bola_{i}.jpg', cv.IMREAD_GRAYSCALE)
    
    # Diferença correspondente
    diferenca = cv.imread(f"img_ex3_3/bola_diff_{i}.jpg", cv.IMREAD_GRAYSCALE)  # Substitua com a maneira correta de carregar as diferenças
    
    # Reconstruir a frame original
    frame_original_reconstruida = reconstruir_frame_original(frame_predita, diferenca)
    t1 = time()
    tDecomp = t1 - t0
    TimesDecomp_Bola.append(tDecomp)

    frames_reconstruidas.append(frame_original_reconstruida)
    
    cv.imwrite(f'img_ex3_3/bola_reconstruida_{i}.jpg', frame_original_reconstruida)

    plt.figure(figsize=(24, 8))
    
    plt.subplot(141)
    plt.axis("off")
    plt.imshow(frame_referencia, cmap='gray')
    plt.title('Frame de Referência (I-Frame)')
    
    plt.subplot(142)
    plt.axis("off")
    plt.imshow(frame_predita, cmap='gray')
    plt.title('Frame Predita')
    
    plt.subplot(143)
    plt.axis("off")
    plt.imshow(frame_original_reconstruida, cmap='gray')
    plt.title('Frame Reconstruída')

    plt.subplot(144)
    plt.axis("off")
    plt.imshow(frame_original, cmap='gray')
    plt.title('Frame Original')
    
    plt.show()

print("|    Frames    |  Tempo (Decomp) (s)  |")
print("|--------------|----------------------|")

# Preenchendo a tabela com os valores dos arrays
for frames_idx, tempoD in zip(Frames_IDX, TimesDecomp_Bola):
    frame_str = f"{frames_idx+1}".center(12)
    tempoD_str = f"{tempoD:.7f}".center(20)
    print(f"| {frame_str} | {tempoD_str} |")

print("\n\nFim da Tabela")


In [ ]:
# Carregar a I-frame de referência
frame_referencia = cv.imread('carro_1.tiff', cv.IMREAD_GRAYSCALE)

frames_reconstruidas = []
Frames_IDX = []
TimesDecomp_Carro = []
# Loop para reconstruir as frames originais
for i in range(1, 11):

    frame_original = cv.imread(f'carro_{i+1}.tiff', cv.IMREAD_GRAYSCALE)

    Frames_IDX.append(i)
    t0 = time()
    # Carregar a frame predita
    frame_predita = cv.imread(f'img_ex3_3/carro_{i}.jpg', cv.IMREAD_GRAYSCALE)
    
    # Diferença correspondente
    diferenca = cv.imread(f"img_ex3_3/carro_diff_{i}.jpg", cv.IMREAD_GRAYSCALE)
    
    # Reconstruir a frame original
    frame_original_reconstruida = reconstruir_frame_original(frame_predita, diferenca)
    t1= time()
    tDecomp = t1 - t0
    TimesDecomp_Carro.append(tDecomp)

    frames_reconstruidas.append(frame_original_reconstruida)
    
    cv.imwrite(f'img_ex3_3/carro_reconstruido_{i}.jpg', frame_original_reconstruida)

    plt.figure(figsize=(24, 8))
    
    plt.subplot(141)
    plt.imshow(frame_referencia, cmap='gray')
    plt.title('Frame de Referência (I-Frame)')
    
    plt.subplot(142)
    plt.axis("off")
    plt.imshow(frame_predita, cmap='gray')
    plt.title('Frame Predita')

    plt.subplot(143)
    plt.axis("off")
    plt.imshow(frame_original_reconstruida, cmap='gray')
    plt.title('Frame Reconstruída')

    plt.subplot(143)
    plt.axis("off")
    plt.imshow(frame_original_reconstruida, cmap='gray')
    plt.title('Frame Reconstruída')
    
    plt.subplot(144)
    plt.axis("off")
    plt.imshow(frame_original, cmap='gray')
    plt.title('Frame Original')

    

    plt.show()

print("|    Frames    |  Tempo (Decomp) (s)  |")
print("|--------------|----------------------|")

# Preenchendo a tabela com os valores dos arrays
for frames_idx, tempoD in zip(Frames_IDX, TimesDecomp_Carro):
    frame_str = f"{frames_idx+1}".center(12)
    tempoD_str = f"{tempoD:.7f}".center(20)
    print(f"| {frame_str} | {tempoD_str} |")

print("\n\nFim da Tabela")